In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import pandas as pd

In [2]:
eval_interval = 300
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200

batch_size = 64
block_size = 256
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
max_iters = 5000

torch.manual_seed(42)

In [3]:
with open('felatun_bey_tam_metin.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

In [5]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [6]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [7]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


In [8]:
print(f"Cihaz: {device}, Vocab Size: {vocab_size}, n_embd: {n_embd}")

Cihaz: cuda, Vocab Size: 94, n_embd: 384


### Self-Attention Head Sınıfı

In [9]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)

        # Attention score
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5) # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)


        # ağırlıklı toplam
        v = self.value(x) # (B, T, head_size)
        out = wei @ v     # (B, T, head_size)
        return out

In [10]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)


    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

In [11]:
class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

### Self-attention'lı Bigram Modeli

In [13]:
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_head = MultiHeadAttention(n_head,n_embd // n_head)
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)


    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B, T, n_embd)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, n_embd)
        x = tok_emb + pos_emb # (B, T, n_embd)

        # Self-attention uygula
        x = self.sa_head(x) # (B, T, n_embd)
        x = self.ln_f(x) # (B,T,C)
        # Logitleri üret
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx: (B, T) boyutunda mevcut context
        for _ in range(max_new_tokens):
            # position embedding sınırını aşmamak için son 8 karaktere bakıyoruz
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] # (B, C)
            probs = F.softmax(logits, dim=-1) # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

### Modelin Eğitilmesi

In [14]:
torch.manual_seed(42)
model = BigramLanguageModel().to(device)
print(f"Toplam parameter: {sum(p.numel() for p in model.parameters()):,}")

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        print(f"{iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

Toplam parameter: 761,566
0: train loss 4.6441, val loss 4.6720
300: train loss 2.2358, val loss 2.2311
600: train loss 2.0395, val loss 2.0441
900: train loss 1.9638, val loss 1.9800
1200: train loss 1.9183, val loss 1.9435
1500: train loss 1.8970, val loss 1.9246
1800: train loss 1.8784, val loss 1.9353
2100: train loss 1.8633, val loss 1.9266
2400: train loss 1.8512, val loss 1.9174
2700: train loss 1.8482, val loss 1.9206
3000: train loss 1.8370, val loss 1.9114
3300: train loss 1.8293, val loss 1.8982
3600: train loss 1.8279, val loss 1.9148
3900: train loss 1.8237, val loss 1.9199
4200: train loss 1.8169, val loss 1.9140
4500: train loss 1.8184, val loss 1.9068
4800: train loss 1.8116, val loss 1.8972
4999: train loss 1.8082, val loss 1.9039


### Tam Transformer Bloğu


In [15]:
class TransformerLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # Final LayerNorm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x) # Pre-LN Transformer blokları
        x = self.ln_f(x)   # Final LayerNorm
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx





In [16]:
torch.manual_seed(42)
model_ln = TransformerLanguageModel().to(device)
print(f"Toplam parameter: {sum(p.numel() for p in model_ln.parameters()):,}")

optimizer = torch.optim.AdamW(model_ln.parameters(), lr=learning_rate)

for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model_ln)
        print(f"{iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model_ln(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

final_losses_ln = estimate_loss(model_ln)
train_loss_ln = final_losses_ln['train']
val_loss_ln = final_losses_ln['val']


Toplam parameter: 10,811,230
0: train loss 4.7122, val loss 4.7094
300: train loss 2.0518, val loss 2.0608
600: train loss 1.3973, val loss 1.5260
900: train loss 1.1233, val loss 1.4111
1200: train loss 0.9169, val loss 1.4336
1500: train loss 0.7242, val loss 1.4864
1800: train loss 0.5548, val loss 1.6173
2100: train loss 0.4046, val loss 1.7389
2400: train loss 0.2947, val loss 1.9083
2700: train loss 0.2192, val loss 2.0407
3000: train loss 0.1709, val loss 2.1885
3300: train loss 0.1389, val loss 2.3304
3600: train loss 0.1196, val loss 2.4385
3900: train loss 0.1082, val loss 2.5527
4200: train loss 0.0999, val loss 2.6142
4500: train loss 0.0947, val loss 2.6877
4800: train loss 0.0890, val loss 2.8160
4999: train loss 0.0877, val loss 2.8318


In [17]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_text_ln = decode(model_ln.generate(context, max_new_tokens=500)[0].tolist())
print(generated_text_ln)



2289 gerçekten
2290 kalk ışağı, dertle
2291 doğuruluğunu sevişmek
2292 karşılamak
2293 neşilde
2294 inanılmış 
2295 nonaylanmış, doğruluğu kabul edilmiş
2296 ağlamak
2297 gönül yazım, dilmek
2298 fazlasıyla bulunanlar
2299 kız kardeş
Anadolu Üniversitesi
198
Can - ( alım) Siz biraz hangi severim. Dadı Kalfa biz de ben kendi-
860 dostlar
861 ortaya çıkarmak, tabi olmak
862 okuyuşlu şekli
863 cimrilik, haspat
864 uymak; sebep
865 söz
Anadolu sevmek
866 koymak, öyle olduğunu kabul etmek 
867 hakika
